In [ ]:
# ~/Documents/drake-iiwa-driver/bazel-bin/kuka-driver/kuka_driver --fri_ip 192.170.10.2 --fri_port 30200
# ~/Documents/drake-iiwa-driver/bazel-bin/kuka-driver/kuka_driver --fri_ip 192.170.10.3 --fri_port 30201 --lcm_command_channel "IIWA_COMMAND_2" --lcm_status_channel "IIWA_STATUS_2"

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
from pydrake.all import (
    DiagramBuilder,
    Demultiplexer,
    Multiplexer,
    Simulator,
    Sine,
)
from iiwa_hardware import (
    MakePlanarBimanualStation,
)

In [ ]:
builder = DiagramBuilder()

station = builder.AddSystem(MakePlanarBimanualStation())

sine = builder.AddSystem(
    Sine(
        amplitude=np.pi/2,
        frequency=np.pi / 10,  # rad/s
        phase=0.0,
        size=1,
    )
)

for k in range(2):
    demux = builder.AddSystem(
        Demultiplexer(output_ports_sizes=[2, 1])
    )
    builder.Connect(station.get_output_port(k), demux.get_input_port())

    mux = builder.AddSystem(
        Multiplexer(input_sizes=[2, 1])
    )
    builder.Connect(demux.get_output_port(0), mux.get_input_port(0))
    builder.Connect(sine.get_output_port(0), mux.get_input_port(1))
    builder.Connect(mux.get_output_port(), station.get_input_port(k))

diagram = builder.Build()
simulator = Simulator(diagram)

simulator.set_target_realtime_rate(1.0)
simulator.AdvanceTo(1000)